# Hamiltonian mechanics: phase space and symplectic integration

*mechanics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/hamiltonian_phase_space)

Write the pendulum Hamiltonian $H(q, p)$ once and let torch.autograd give Hamilton's equations $\dot z_a = J_{ab}\,\partial H/\partial z_b$. A whole fan of orbits and a Liouville blob move as one tensor, and the symplectic leapfrog is raced against explicit Euler and RK4 over many periods.

**Vector form**

$$\dot q = \frac{\partial H}{\partial p},\quad \dot p = -\frac{\partial H}{\partial q},\qquad H(q, p) = \frac{p^2}{2mL^2} - mgL\cos q$$

**Index (tensor) form**

$$\dot z_{ba} = J_{ac}\,\frac{\partial H}{\partial z_{bc}},\qquad J = \begin{pmatrix} 0 & I \\ -I & 0 \end{pmatrix},\qquad A = \tfrac12\sum_k \big(q_k\,p_{k+1} - q_{k+1}\,p_k\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $z_{ba} = (q_b, p_b)$ | `z[b, :]` | (B, 2) | phase-space point $b$: angle $q$ and angular momentum $p$; every orbit and blob point is one row |
| $H(z)$ | `pendulum(m, L, g)(z)` | (B,) | the Hamiltonian, the only physics typed in by hand |
| $\partial H/\partial z_{bc}$ | `autograd.grad(H(z).sum(), z)` | (B, 2) | gradient of the sum gives each independent point its own gradient |
| $J_{ac}$ | `symplectic_matrix(1)` | (2, 2) | symplectic matrix $[[0, I], [-I, 0]]$: turns the gradient by 90° |
| $J_{ac}\,\partial H/\partial z_{bc}$ | `einsum('ab,...b->...a', J, dH)` | (B, 2) | repeated $c$ ⇒ summed: Hamilton's equations for all points at once |
| $p \leftarrow p - \tfrac12\Delta t\,\partial_q H$ | `leapfrog_step(H, z, dt)` | (B, 2) | kick–drift–kick (Störmer–Verlet): each sub-step is a shear, so area is kept exactly |
| $z \leftarrow z + \Delta t\,\dot z$ | `euler_step(H, z, dt)` | (B, 2) | explicit Euler, for comparison: stretches phase space every step |
| $A$ | `shoelace_area(blob)` | (T,) | area of the blob polygon; Liouville says it stays constant |

The separatrix $H = mgL$, i.e. $p = \pm 2mL\sqrt{gL}\cos(q/2)$, divides librations (closed loops) from rotations (wavy lines that leave on the right and come back on the left). Near it the period grows without bound, so the blob's inner edge laps its outer edge: it stretches into a thin spiral but its area stays the same (Liouville's theorem). Leapfrog is only second order, yet its energy error stays bounded forever because it exactly conserves a nearby "shadow" Hamiltonian; RK4 is far more accurate per step but its error creeps up steadily, and explicit Euler multiplies phase-space area by about $1 + \omega^2\Delta t^2$ per step, so the energy explodes. Try a larger $\Delta t$, or the harmonic system where the blob just rotates rigidly.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/hamiltonian_phase_space/main.py
"""Main loop: parameters → Hamiltonian → initial phase-space points → three integrators → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    m, L, g = params["m"], params["length"], 9.81
    H = equation.pendulum(m, L, g) if params["system"] == "pendulum" else equation.harmonic(m, L, g)
    p_sep = 2 * m * L * math.sqrt(g * L)  # momentum at q = 0 on the separatrix H = mgL

    z0 = {
        "orbits": tensors.orbit_fan(params["n_orbits"], p_sep, dev),  # (B, 2) phase portrait
        "test": tensors.test_orbit(params["amplitude"], dev),  # (1, 2) energy comparison
        "blob": tensors.blob((0.0, 0.6 * p_sep), (0.3, 0.1 * p_sep), device=dev),  # (K, 2) Liouville blob
    }
    sol = solve.solve(H, z0, params["dt"], params["t_max"])

    scale = m * g * L  # energy unit: the separatrix sits at H = mgL
    energy = {k: H(tr["test"][:, 0]) for k, tr in sol["traj"].items()}  # (T,) per integrator
    area = {k: equation.shoelace_area(tr["blob"]) for k, tr in sol["traj"].items()}  # (T,) per integrator

    print(f"separatrix energy mgL = {scale:.3f} J, momentum at the bottom 2mL√(gL) = {p_sep:.3f} kg·m²/s")
    period0 = 2 * math.pi * math.sqrt(L / g)
    measured = solve.small_oscillation_period(H, math.radians(2.0), 4.2 * period0, device=dev)
    print(f"small-oscillation period (2° swing): {measured:.4f} s   2π√(L/g) = {period0:.4f} s")
    one_second = int(round(1.0 / params["dt"]))
    for k in solve.STEPPERS:
        err = ((energy[k] - energy[k][0]) / scale).abs()
        ratio = area[k] / area[k][0]
        end = f"{ratio[-1]:.5f}" if k != "Euler" else "too stretched to measure"
        print(
            f"{k:8s}  max |ΔE|/mgL = {err.max():.2e}  (last {err[-1]:.2e})   "
            f"blob area A/A0: after 1 s {ratio[one_second]:.4f}, at the end {end}"
        )
    return plot.animate(sol, energy, area, scale, p_sep, params["system"], out_dir / "hamiltonian_phase_space.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/hamiltonian_phase_space/equation.py
r"""Physics → tensor translation.

Hamilton's equations for coordinates q_i and momenta p_i,

    \dot q_i = \frac{\partial H}{\partial p_i}, \qquad \dot p_i = -\frac{\partial H}{\partial q_i},

become one equation for the phase-space point z = (q, p) with the symplectic matrix J = [[0, I], [−I, 0]]:

    \dot z_a = J_{ab} \frac{\partial H}{\partial z_b}        (repeated b ⇒ summed)

H is typed in once as a function; torch.autograd supplies ∂H/∂z_b. For the pendulum (q = angle from the
downward vertical, p = angular momentum)

    H(q, p) = \frac{p^2}{2 m L^2} - m g L \cos q ,

and the separatrix H = mgL divides swinging back and forth (libration) from going over the top (rotation).
Liouville's theorem: the flow keeps phase-space area, measured here with the shoelace formula

    A = \tfrac12 \sum_k (q_k p_{k+1} - q_{k+1} p_k)        (k runs around the blob's boundary).
"""

from typing import Callable

import torch

Hamiltonian = Callable[[torch.Tensor], torch.Tensor]  # H(z (..., 2n)) → (...)


def pendulum(m: float, L: float, g: float) -> Hamiltonian:
    """H(q, p) = p²/(2mL²) − mgL cos q, with z[..., 0] = q and z[..., 1] = p."""

    def H(z: torch.Tensor) -> torch.Tensor:
        q, p = z[..., 0], z[..., 1]
        return p**2 / (2 * m * L**2) - m * g * L * torch.cos(q)

    return H


def harmonic(m: float, L: float, g: float) -> Hamiltonian:
    """The small-angle pendulum H = p²/(2mL²) + ½ mgL q² (constant −mgL dropped): a harmonic oscillator."""

    def H(z: torch.Tensor) -> torch.Tensor:
        q, p = z[..., 0], z[..., 1]
        return p**2 / (2 * m * L**2) + 0.5 * m * g * L * q**2

    return H


def symplectic_matrix(n: int, dtype=torch.float64, device=None) -> torch.Tensor:
    """J = [[0, I], [−I, 0]] of shape (2n, 2n)."""
    eye = torch.eye(n, dtype=dtype, device=device)
    zero = torch.zeros(n, n, dtype=dtype, device=device)
    return torch.cat([torch.cat([zero, eye], dim=1), torch.cat([-eye, zero], dim=1)], dim=0)


def grad_H(H: Hamiltonian, z: torch.Tensor) -> torch.Tensor:
    """∂H/∂z_b by autograd for a whole batch of points at once.   (..., 2n) → (..., 2n)

    The points are independent, so the gradient of the sum ΣH gives every point's own gradient.
    """
    z = z.detach().requires_grad_(True)
    (dH,) = torch.autograd.grad(H(z).sum(), z)
    return dH


def hamilton_rhs(H: Hamiltonian, z: torch.Tensor) -> torch.Tensor:
    r"""\dot z_a = J_{ab} ∂H/∂z_b (b summed).   (..., 2n) → (..., 2n)"""
    J = symplectic_matrix(z.shape[-1] // 2, z.dtype, z.device)  # (2n, 2n)
    return torch.einsum("ab,...b->...a", J, grad_H(H, z))


def pendulum_rhs_by_hand(z: torch.Tensor, m: float, L: float, g: float) -> torch.Tensor:
    """The same equations derived on paper: q̇ = p/(mL²), ṗ = −mgL sin q (for checking autograd)."""
    q, p = z[..., 0], z[..., 1]
    return torch.stack([p / (m * L**2), -m * g * L * torch.sin(q)], dim=-1)


def separatrix_momentum(q: torch.Tensor, m: float, L: float, g: float) -> torch.Tensor:
    """Upper branch of H = mgL:  p = 2 m L √(gL) cos(q/2).   (K,) → (K,)"""
    return 2 * m * L * (g * L) ** 0.5 * torch.cos(q / 2)


def shoelace_area(z: torch.Tensor) -> torch.Tensor:
    r"""Area enclosed by a closed polygon of phase-space points: (..., K, 2) → (...).

    A = ½ Σ_k (q_k p_{k+1} − q_{k+1} p_k); torch.roll gives the next point k + 1 (wrapping K → 0).
    """
    q, p = z[..., 0], z[..., 1]
    q_next, p_next = torch.roll(q, -1, dims=-1), torch.roll(p, -1, dims=-1)
    return 0.5 * (q * p_next - q_next * p).sum(dim=-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/hamiltonian_phase_space/tensors.py
"""Build the concrete tensors: a fan of orbits, the integrator test orbit and the Liouville blob.

Every phase-space point is a row z = (q, p); float64 so that energy errors come from the integrator, not rounding.
"""

import math

import torch

DTYPE = torch.float64


def orbit_fan(n: int, p_sep: float, device=None) -> torch.Tensor:
    """n orbits started at q = 0 with momenta up to 1.5 × the separatrix momentum: (n, 2).

    |p| < p_sep ⇒ libration (swinging), |p| > p_sep ⇒ rotation (going over the top).
    """
    p0 = torch.linspace(0.15, 1.5, n, dtype=DTYPE, device=device) * p_sep  # (n,)
    p0 = p0 * torch.where(torch.arange(n, device=device) % 2 == 0, 1.0, -1.0)  # alternate the direction
    return torch.stack([torch.zeros_like(p0), p0], dim=1)


def test_orbit(amplitude_deg: float, device=None) -> torch.Tensor:
    """One pendulum released from rest at the given angle: (1, 2)."""
    return torch.tensor([[math.radians(amplitude_deg), 0.0]], dtype=DTYPE, device=device)


def blob(center: tuple[float, float], radius: tuple[float, float], k: int = 2000, device=None) -> torch.Tensor:
    """K points on the boundary of a small ellipse in the (q, p) plane, counter-clockwise: (K, 2).

    Many points, because the boundary stretches into a long thin spiral and the polygon must still follow it.
    """
    phi = torch.linspace(0, 2 * math.pi, k + 1, dtype=DTYPE, device=device)[:-1]  # (K,)
    q = center[0] + radius[0] * torch.cos(phi)
    p = center[1] + radius[1] * torch.sin(phi)
    return torch.stack([q, p], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/hamiltonian_phase_space/solve.py
"""Three time steppers for Hamilton's equations: symplectic leapfrog, explicit Euler and RK4.

All take a batch of phase-space points z of shape (..., 2n) and get ∂H/∂z from autograd.
"""

from typing import Callable

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import Hamiltonian, grad_H, hamilton_rhs

Stepper = Callable[[Hamiltonian, torch.Tensor, float], torch.Tensor]


def leapfrog_step(H: Hamiltonian, z: torch.Tensor, dt: float) -> torch.Tensor:
    """Störmer–Verlet (kick–drift–kick) for a separable H = T(p) + V(q):

    p ← p − ½Δt ∂H/∂q,   q ← q + Δt ∂H/∂p,   p ← p − ½Δt ∂H/∂q.
    Each sub-step is a shear of phase space, so the whole step keeps area exactly (it is symplectic).
    """
    n = z.shape[-1] // 2
    q, p = z[..., :n], z[..., n:]
    p = p - 0.5 * dt * grad_H(H, torch.cat([q, p], -1))[..., :n]  # kick
    q = q + dt * grad_H(H, torch.cat([q, p], -1))[..., n:]  # drift
    p = p - 0.5 * dt * grad_H(H, torch.cat([q, p], -1))[..., :n]  # kick
    return torch.cat([q, p], -1)


def euler_step(H: Hamiltonian, z: torch.Tensor, dt: float) -> torch.Tensor:
    """Explicit Euler z ← z + Δt ż: stretches phase space a little every step, so energy grows."""
    return z + dt * hamilton_rhs(H, z)


def rk4_hamilton_step(H: Hamiltonian, z: torch.Tensor, dt: float) -> torch.Tensor:
    """Classical 4th-order Runge–Kutta: very accurate per step, but not symplectic (slow energy drift)."""
    return rk4_step(lambda y, t: hamilton_rhs(H, y), z, 0.0, dt)


STEPPERS: dict[str, Stepper] = {"leapfrog": leapfrog_step, "RK4": rk4_hamilton_step, "Euler": euler_step}


def integrate(stepper: Stepper, H: Hamiltonian, z0: torch.Tensor, dt: float, n_steps: int, label: str = "steps"):
    """Whole trajectory, shape (n_steps + 1, *z0.shape)."""
    zs = [z0]
    z = z0
    for _ in steps(n_steps, label):
        z = stepper(H, z, dt)
        zs.append(z)
    return torch.stack(zs)


def solve(H: Hamiltonian, z0: dict[str, torch.Tensor], dt: float, t_max: float) -> dict:
    """Run every initial-condition set with every integrator, all points of a set as one batch.

    z0 maps a name ("orbits", "test", "blob") to a (B, 2) tensor; each set is stacked into one batch
    per integrator so a single step moves all B·(sets) points. Returns times (T,) and
    traj[integrator][name] of shape (T, B, 2).
    """
    n_steps = int(round(t_max / dt))
    names = list(z0)
    sizes = [z0[k].shape[0] for k in names]
    batch = torch.cat([z0[k] for k in names])  # (ΣB, 2) every point in one tensor
    traj = {}
    for label, stepper in STEPPERS.items():
        path = integrate(stepper, H, batch, dt, n_steps, f"{label} steps")  # (T, ΣB, 2)
        traj[label] = dict(zip(names, torch.split(path, sizes, dim=1)))
    times = torch.arange(n_steps + 1, dtype=batch.dtype, device=batch.device) * dt
    return {"times": times, "traj": traj}


def small_oscillation_period(H: Hamiltonian, amplitude: float, t_max: float, dt: float = 0.01, device=None) -> float:
    """Release one pendulum from rest at a small angle (radians), leapfrog it and measure its period."""
    z0 = torch.tensor([[amplitude, 0.0]], dtype=torch.float64, device=device)
    n_steps = int(round(t_max / dt))
    q = integrate(leapfrog_step, H, z0, dt, n_steps, "small swing")[:, 0, 0]  # (T,)
    return period(torch.arange(n_steps + 1, dtype=q.dtype, device=q.device) * dt, q)


def period(times: torch.Tensor, q: torch.Tensor) -> float:
    """Mean period of q(t) from its downward zero crossings (linear interpolation between steps)."""
    down = torch.nonzero((q[:-1] > 0) & (q[1:] <= 0)).flatten()  # steps just before a crossing
    t_cross = times[down] + (times[1] - times[0]) * q[down] / (q[down] - q[down + 1])
    return float((t_cross[-1] - t_cross[0]) / (len(t_cross) - 1))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/hamiltonian_phase_space/plot.py
"""Animate the phase portrait (orbits, separatrix, Liouville blob) next to energy error and blob area per integrator."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

COLORS = {"leapfrog": ACCENT[0], "RK4": ACCENT[2], "Euler": ACCENT[1]}


def wrap(q: np.ndarray) -> np.ndarray:
    """Angle into [−π, π), with NaN where an orbit jumps across the edge (so no line is drawn across)."""
    w = (q + math.pi) % (2 * math.pi) - math.pi
    jumps = np.abs(np.diff(w, axis=0)) > math.pi
    w[1:][jumps] = np.nan
    return w


def animate(sol: dict, energy: dict, area: dict, scale: float, p_sep: float, system: str, path: Path) -> Path:
    t = sol["times"].cpu().numpy()  # (T,)
    leap = sol["traj"]["leapfrog"]
    orbits = leap["orbits"].cpu().numpy()  # (T, B, 2)
    blob = leap["blob"].cpu().numpy()  # (T, K, 2)
    pendulum = system == "pendulum"
    q_orb = wrap(orbits[..., 0]) if pendulum else orbits[..., 0]
    p_orb = orbits[..., 1]
    err = {k: np.abs((e - e[0]).cpu().numpy()) / scale + 1e-16 for k, e in energy.items()}  # (T,) each
    ratio = {k: (a / a[0]).cpu().numpy() for k, a in area.items()}  # (T,) each
    for r in ratio.values():  # once a blob leaves the scale (Euler) its polygon is too stretched to measure
        off = np.nonzero(np.abs(r - 1) > 0.5)[0]
        if len(off):
            r[off[0] + 1 :] = np.nan

    fig = plt.figure(figsize=(8, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.35, 1])
    ax = fig.add_subplot(grid[:, 0])
    ax_e = fig.add_subplot(grid[0, 1])
    ax_a = fig.add_subplot(grid[1, 1], sharex=ax_e)

    # --- left: phase plane (q, p) ---
    q_lim = math.pi if pendulum else 1.1 * np.nanmax(np.abs(q_orb))
    ax.set_xlim(-q_lim, q_lim)
    ax.set_ylim(-1.65 * p_sep, 1.65 * p_sep)
    ax.set_xlabel("q (rad)")
    ax.set_ylabel("p (kg·m²/s)")
    if pendulum:
        qs = np.linspace(-math.pi, math.pi, 200)
        ps = p_sep * np.cos(qs / 2)  # separatrix p = ±2mL√(gL) cos(q/2)
        for sign in (1, -1):
            ax.plot(qs, sign * ps, color="#f59e0b", lw=2.2, label="separatrix $H = mgL$" if sign > 0 else None)
    lines = [ax.plot([], [], lw=0.9, color="#6b7280")[0] for _ in range(orbits.shape[1])]
    heads = ax.plot([], [], "o", ms=3, color="#374151")[0]
    ax.fill(blob[0, :, 0], blob[0, :, 1], fill=False, ls="--", lw=0.8, color=ACCENT[0])
    patch = ax.fill(blob[0, :, 0], blob[0, :, 1], color=ACCENT[0], alpha=0.55, label="blob (leapfrog)")[0]
    clock = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top", fontsize=9)
    ax.legend(loc="lower right", fontsize=7, framealpha=0.9)
    ax.set_title("phase space: libration, rotation, Liouville blob", fontsize=9)

    # --- right top: energy error of the test orbit ---
    ax_e.set_yscale("log")
    ax_e.set_xlim(t[0], t[-1])
    ax_e.set_ylim(1e-9, 1e2)
    ax_e.set_yticks([1e-9, 1e-6, 1e-3, 1e0])
    ax_e.minorticks_off()
    ax_e.set_ylabel("|ΔE| / mgL", fontsize=8)
    e_lines = {k: ax_e.plot([], [], lw=0.9, color=c, label=k)[0] for k, c in COLORS.items()}
    ax_e.legend(loc="upper left", fontsize=7, ncol=3)

    # --- right bottom: blob area ---
    ax_a.set_ylim(0.9, 1.5)
    ax_a.axhline(1.0, color="#9ca3af", lw=0.8, ls="--")
    ax_a.set_ylabel("area A / A₀", fontsize=8)
    ax_a.set_xlabel("t (s)")
    a_lines = {k: ax_a.plot([], [], lw=1.2, color=c, zorder=3 if k == "leapfrog" else 2)[0] for k, c in COLORS.items()}
    ax_a.text(
        0.98,
        0.92,
        "Euler: area keeps growing ↑",
        transform=ax_a.transAxes,
        ha="right",
        va="top",
        fontsize=7,
        color=COLORS["Euler"],
    )
    plt.setp(ax_e.get_xticklabels(), visible=False)

    def update(step: int):
        for b, line in enumerate(lines):
            line.set_data(q_orb[: step + 1, b], p_orb[: step + 1, b])
        heads.set_data(q_orb[step], p_orb[step])
        patch.set_xy(blob[step])
        for k in COLORS:
            e_lines[k].set_data(t[: step + 1], err[k][: step + 1])
            a_lines[k].set_data(t[: step + 1], ratio[k][: step + 1])
        clock.set_text(f"t = {t[step]:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path, fps=12)  # ~10 s: slow enough to follow the blob


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'system': 'pendulum',
    'm': 1.0,
    'length': 1.0,
    'amplitude': 120.0,
    'n_orbits': 12,
    'dt': 0.05,
    't_max': 50.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.hamiltonian_phase_space")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.hamiltonian_phase_space.main")
problem_dir = Path("problems/mechanics/hamiltonian_phase_space")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
